# Argus vs Zizmor: Security Findings Comparison

This notebook compares security findings from **Argus** and **Zizmor** tools.

**Focus:** Detection capabilities and coverage analysis

**Tools:**
- **Argus**: Taint analysis focused on data flow tracking
- **Zizmor**: Static analysis for GitHub Actions security best practices

## 1. Setup & Dependencies

In [ ]:
import subprocess
import sys

packages = ['pandas', 'matplotlib', 'seaborn', 'plotly']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

In [ ]:
import json
import os
from pathlib import Path
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

BASE_DIR = Path('../results/sarif_outputs')

print(f"Base SARIF directory: {BASE_DIR}")

## 2. Helper Functions

In [ ]:
def load_sarif_file(filepath):
    with open(filepath, 'r') as f:
        return json.load(f)

def extract_findings(sarif_data, tool_name):
    findings = []
    for run in sarif_data.get('runs', []):
        for result in run.get('results', []):
            locations = result.get('locations', [])
            location_info = {}
            
            if locations:
                loc = locations[0]
                phys_loc = loc.get('physicalLocation', {})
                artifact = phys_loc.get('artifactLocation', {})
                region = phys_loc.get('region', {})
                
                location_info['file'] = artifact.get('uri', 'unknown')
                location_info['line'] = region.get('startLine', 'unknown')
                location_info['column'] = region.get('startColumn', 'unknown')
            
            message = result.get('message', {}).get('text', '')
            rule_id = result.get('ruleId', 'unknown')
            level = result.get('level', 'unknown')
            kind = result.get('kind', '')
            properties = result.get('properties', {})
            
            finding = {
                'tool': tool_name,
                'rule_id': rule_id,
                'message': message,
                'level': level,
                'kind': kind,
                'file': location_info.get('file', 'unknown'),
                'line': location_info.get('line', 'unknown'),
                'column': location_info.get('column', 'unknown'),
                'properties': properties
            }
            findings.append(finding)
    return findings

In [ ]:
def categorize_finding(rule_id, message, tool):
    rule_lower = rule_id.lower()
    message_lower = message.lower()
    
    if tool == 'argus':
        if 'ContextToSink' in rule_id or 'Taint' in rule_id:
            return 'template_injection'
        if 'Secret' in rule_id:
            return 'credentials'
    
    if tool == 'zizmor':
        if 'template-injection' in rule_lower:
            return 'template_injection'
        if 'excessive-permissions' in rule_lower:
            return 'permissions'
        if 'unpinned' in rule_lower:
            return 'unpinned_action'
        if 'credentials' in rule_lower or 'secret' in rule_lower:
            return 'credentials'
        if 'artifact' in rule_lower:
            return 'artifacts'
    
    keywords = {
        'template_injection': ['template', 'injection', 'github.event', 'taint', 'contextto'],
        'permissions': ['permission', 'excessive', 'least privilege'],
        'unpinned_action': ['pin', 'hash', 'unpinned', 'version'],
        'credentials': ['secret', 'credential', 'token', 'password'],
        'artifacts': ['artifact', 'download', 'upload']
    }
    
    for cat, kws in keywords.items():
        for kw in kws:
            if kw in message_lower or kw in rule_lower:
                return cat
    return 'other'

def get_severity_mapping(level):
    return {'error': 3, 'warning': 2, 'note': 1, 'none': 0, 'unknown': 0}.get(level, 0)

## 3. Load All SARIF Data

In [ ]:
workflows = [d.name for d in BASE_DIR.iterdir() if d.is_dir()] if BASE_DIR.exists() else []
print(f"Found {len(workflows)} workflows:")
for w in sorted(workflows)[:5]:
    print(f"  - {w}")

In [ ]:
all_findings = []
workflow_data = {}

for workflow_name in workflows:
    workflow_dir = BASE_DIR / workflow_name
    
    argus_file = workflow_dir / 'argus.sarif'
    argus_findings = []
    if argus_file.exists():
        try:
            argus_data = load_sarif_file(argus_file)
            argus_findings = extract_findings(argus_data, 'argus')
            for f in argus_findings:
                f['category'] = categorize_finding(f['rule_id'], f['message'], 'argus')
                f['workflow'] = workflow_name
                f['severity_score'] = get_severity_mapping(f['level'])
        except Exception as e:
            print(f"Error loading {argus_file}: {e}")
    
    zizmor_file = workflow_dir / 'zizmor.sarif'
    zizmor_findings = []
    if zizmor_file.exists():
        try:
            zizmor_data = load_sarif_file(zizmor_file)
            zizmor_findings = extract_findings(zizmor_data, 'zizmor')
            for f in zizmor_findings:
                f['category'] = categorize_finding(f['rule_id'], f['message'], 'zizmor')
                f['workflow'] = workflow_name
                f['severity_score'] = get_severity_mapping(f['level'])
        except Exception as e:
            print(f"Error loading {zizmor_file}: {e}")
    
    workflow_data[workflow_name] = {'argus': argus_findings, 'zizmor': zizmor_findings}
    all_findings.extend(argus_findings)
    all_findings.extend(zizmor_findings)

print(f"\nTotal findings loaded: {len(all_findings)}")

In [ ]:
df = pd.DataFrame(all_findings)
print(f"DataFrame shape: {df.shape}")
print(f"\nTools: {df['tool'].unique()}")
print(f"Categories: {df['category'].unique()}")

## 4. Overview Statistics

In [ ]:
total_argus = len(df[df['tool'] == 'argus'])
total_zizmor = len(df[df['tool'] == 'zizmor'])

print("=" * 60)
print("OVERVIEW STATISTICS")
print("=" * 60)
print(f"\nTotal Workflows Analyzed: {len(workflows)}")
print(f"\nTotal Findings:")
print(f"  Argus:  {total_argus}")
print(f"  Zizmor: {total_zizmor}")

In [ ]:
summary_data = []
for workflow_name in workflows:
    w_data = workflow_data[workflow_name]
    summary_data.append({
        'workflow': workflow_name,
        'argus_count': len(w_data['argus']),
        'zizmor_count': len(w_data['zizmor']),
        'argus_categories': len(set(f['category'] for f in w_data['argus'])),
        'zizmor_categories': len(set(f['category'] for f in w_data['zizmor']))
    })

summary_df = pd.DataFrame(summary_data)
print("\nWorkflow Summary:")
print(summary_df.to_string(index=False))

## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax1 = axes[0, 0]
tools = ['Argus', 'Zizmor']
counts = [total_argus, total_zizmor]
colors = ['#3498db', '#e74c3c']
bars = ax1.bar(tools, counts, color=colors, edgecolor='black', linewidth=1.2)
ax1.set_ylabel('Number of Findings', fontsize=12)
ax1.set_title('Total Findings by Tool', fontsize=14, fontweight='bold')
for bar, count in zip(bars, counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             str(count), ha='center', va='bottom', fontsize=14, fontweight='bold')

ax2 = axes[0, 1]
x = range(len(workflows))
width = 0.35
argus_counts = summary_df['argus_count'].tolist()
zizmor_counts = summary_df['zizmor_count'].tolist()
ax2.bar([i - width/2 for i in x], argus_counts, width, label='Argus', color='#3498db', edgecolor='black')
ax2.bar([i + width/2 for i in x], zizmor_counts, width, label='Zizmor', color='#e74c3c', edgecolor='black')
ax2.set_xlabel('Workflow', fontsize=12)
ax2.set_ylabel('Number of Findings', fontsize=12)
ax2.set_title('Findings per Workflow', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels([w.replace('vwbench_workflow', 'WF') for w in workflows], rotation=45, ha='right')
ax2.legend()

ax3 = axes[1, 0]
category_counts = df.groupby(['tool', 'category']).size().unstack(fill_value=0)
category_counts.plot(kind='bar', ax=ax3, color=sns.color_palette('husl', len(category_counts.columns)), edgecolor='black')
ax3.set_xlabel('Tool', fontsize=12)
ax3.set_ylabel('Number of Findings', fontsize=12)
ax3.set_title('Findings by Category', fontsize=14, fontweight='bold')
ax3.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
ax3.set_xticklabels(['Argus', 'Zizmor'], rotation=0)

ax4 = axes[1, 1]
for i, tool in enumerate(['argus', 'zizmor']):
    tool_df = df[df['tool'] == tool]
    sev_counts = tool_df.groupby('level').size()
    ax4.bar([i*3 + j for j in range(len(sev_counts))], sev_counts.values, 
            label=tool.capitalize(), color=colors[i], alpha=0.8)
ax4.set_xticks([1, 4])
ax4.set_xticklabels(['Error', 'Warning'])
ax4.set_ylabel('Count')
ax4.set_title('Findings by Severity', fontsize=14, fontweight='bold')
ax4.legend()

plt.tight_layout()
plt.savefig('findings_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved: findings_overview.png")

## 6. Category Analysis

In [ ]:
print("=" * 70)
print("CATEGORY BREAKDOWN")
print("=" * 70)

for tool in ['argus', 'zizmor']:
    print(f"\n{tool.upper()}:")
    tool_df = df[df['tool'] == tool]
    cat_counts = tool_df.groupby('category').size().sort_values(ascending=False)
    for cat, count in cat_counts.items():
        pct = count / len(tool_df) * 100 if len(tool_df) > 0 else 0
        print(f"  {cat:25s}: {count:4d} ({pct:5.1f}%)")

## 7. Rule ID Analysis

In [ ]:
print("=" * 70)
print("RULE ID DISTRIBUTION")
print("=" * 70)

for tool in ['argus', 'zizmor']:
    print(f"\n{tool.upper()} - Top Rules:")
    tool_df = df[df['tool'] == tool]
    rule_counts = tool_df.groupby('rule_id').size().sort_values(ascending=False)
    for rule, count in rule_counts.head(8).items():
        sample = tool_df[tool_df['rule_id'] == rule]['message'].iloc[0][:50] if len(tool_df[tool_df['rule_id'] == rule]) > 0 else ''
        print(f"  {rule:40s}: {count:3d}x  ({sample}...)")

## 8. Coverage Matrix

In [ ]:
print("=" * 80)
print("TOOL COVERAGE MATRIX")
print("=" * 80)

categories = ['template_injection', 'permissions', 'unpinned_action', 'credentials']

header = f"{'Workflow':25s}" + "".join(f"{c[:8]:>10s}" for c in categories)
print(header)
print("-" * 80)

for workflow in workflows[:10]:
    w_data = workflow_data[workflow]
    row = workflow[:25]
    for cat in categories:
        argus_has = any(f['category'] == cat for f in w_data['argus'])
        zizmor_has = any(f['category'] == cat for f in w_data['zizmor'])
        if argus_has and zizmor_has:
            marker = "  BOTH  "
        elif argus_has:
            marker = "  ARGUS "
        elif zizmor_has:
            marker = "  ZIZMOR"
        else:
            marker = "   -    "
        row += marker
    print(row)

In [ ]:
summary_df.to_csv('workflow_summary.csv', index=False)
df.to_csv('all_findings.csv', index=False)
print("Exported: workflow_summary.csv")
print("Exported: all_findings.csv")
print("\nAnalysis complete!")